## Chapter 04: Prompt Engineering

### Principles of prompt engineering

#### 1. Clear instructions

In [ ]:
import os
from openai import OpenAI

import config

In [92]:
client = OpenAI(
  api_key=config.DEEPSEEK_API_KEY,
  base_url=("https://api.deepseek.com")
)

In [ ]:
# Set the metaprompt i.e. define how the model should behave
system_message = """
You are an AI assistant that helps humans by generating tutorials given a text.
You will be provided with a text. If the text contains any kind of instructions 
on how to proceed with something, generate a tutorial in a bullet list.
Otherwise, inform the user that the text does not contain any instructions.

Text:
"""

# Set the query i.e. where the user will ask the model its questions
instructions = """
To prepare the known sauce from Genova, Italy, you can start by toasting the pine 
nuts to then coarsely chop them in a kitchen mortar together with basil and garlic.
Then, add half of the oil in the kitchen mortar and season with salt and pepper.
Finally, transfer the pesto to a bowl and stir the grated Parmesan cheese.
"""

In [8]:
response = client.chat.completions.create(
  model="deepseek-v4-pro",
  messages=[
    {"role": "system", "content": system_message},
    {"role": "user", "content": instructions},
	],
  stream=False,
  reasoning_effort="high",
  extra_body={"thinking": {"type": "enabled"}}
)

In [7]:
print(
  response.choices[0].message.content
)

Here's a tutorial on how to prepare the sauce:

- **Toast the pine nuts**: Place the pine nuts in a dry pan over medium heat and toast them lightly, stirring frequently, until golden and fragrant. Let them cool.
- **Crush the ingredients**: In a kitchen mortar, combine the toasted pine nuts, fresh basil leaves, and peeled garlic cloves. Coarsely crush and grind them with the pestle until a paste forms.
- **Incorporate oil and season**: While continuing to mix, gradually pour in half of the olive oil. Season with salt and pepper to taste, blending until well combined.
- **Finish with cheese**: Transfer the pesto from the mortar to a bowl. Add the grated Parmesan cheese and stir thoroughly until the cheese is fully incorporated. Your pesto is ready to use.


In [9]:
# Pass the model another text that doesn't contain any instructions
response = client.chat.completions.create(
  model="deepseek-v4-pro",
  messages=[
    {"role": "system", "content": system_message},
    {"role": "user", "content": "The sun is shining and dogs are running on the beach."},
	]
)

In [10]:
print(
  response.choices[0].message.content
)

The text does not contain any instructions. It is a descriptive statement about a sunny beach scene with dogs.


#### 2. Split complex tasks into subtasks

In [ ]:
system_message = """
You are an AI assistant that summarizes articles.
To complete this task, do the following subtasks:

Read the provided article context comprehensively and identify the main topic and 
key points
Generate a paragraph summary of the current article context that captures the 
essential information and conveys the main idea
Print each step of the process

Article:
"""

article = """
Recurrent neural networks, long short-term memory and gated recurrent neural networks 
in particular, have been firmly established as state of the art approaches in sequence 
modeling and transduction problems such as large language modeling and machine translation. 
Numerous efforts have since continued to push the boundaries of recurrent language 
models and encoder-decoder architectures.
Recurrent models typically factor computation along the symbol positions of the 
input and output sequences. Aligning the positions to steps in computation time, 
they generate a sequence of hidden states ht, as a function of the previous hidden 
state ht-1 and the input for position t. This inherently sequential nature precludes 
parallelization within training examples, which becomes critical at longer sequence 
lengths, as memory constraints limit batching across examples. Recent work has 
achieved significant improvements in computational efficiency through factorization 
tricks and conditional computation, while also improving model performance in case 
of the latter. The fundamental constraint of sequential computation, however, remains.
Attention mechanisms have become an integral part of compelling sequence modeling 
and transduction models in various tasks, allowing modeling of dependencies without 
regard to their distance in the input or output sequences. In all but a few cases, 
however, such attention mechanisms are used in conjuction with a recurrent network.
In this work we propose the Transformer, a model architecture eschewing recurrence 
and instead relying entirely on an attention mechanism to draw global dependencies 
between input and output.
The transformer allows for significantly more parallelization and can reach a new 
state of the art in translation quality after being trained for as little as twelve 
hours on eight P100 GPUs.
"""

In [12]:
response = client.chat.completions.create(
  model="deepseek-v4-pro",
  messages= [
    {"role": "system", "content": system_message},
    {"role": "user", "content": article},
	],
  stream=False,
  reasoning_effort="high",
  extra_body={"thinking": {"type": "enabled"}}
)

In [13]:
print(
  response.choices[0].message.content
)

Step 1: Read and analyze the article context. The article discusses the limitations of recurrent neural networks (RNNs), particularly their sequential computation which hinders parallelization and efficiency for long sequences. It notes that attention mechanisms are typically combined with RNNs to capture dependencies. The key point is the proposal of the Transformer architecture, which eliminates recurrence and relies solely on attention to model global dependencies, resulting in significantly more parallelization and state-of-the-art translation performance.

Step 2: Generate a summary paragraph that conveys the essential information. The summary should state the problem with recurrent models, the existing use of attention, and the novel Transformer approach and its benefits.

Summary: Recurrent neural networks, while dominant in sequence modeling and transduction tasks, are inherently limited by their sequential computation, which restricts parallelization and efficiency for long se

#### 3. Ask for justification

In [14]:
system_message = """
You are an AI assistant specialized in solving riddles.
Given a riddle, solve it the best you can.
Provide a clear justification of your answer and the reasoning behind it.

Riddle:

"""

riddle = """
What has a face and two hands, but no arms or legs?
"""

In [15]:
response = client.chat.completions.create(
  model="deepseek-v4-pro",
  messages=[
    {"role": "system", "content": system_message},
    {"role": "user", "content": riddle},
	],
  stream=False,
  reasoning_effort="high",
  extra_body={"thinking": {"type": "enabled"}}
)

In [16]:
print(
  response.choices[0].message.content
)

A clock.

A clock features a "face" (the dial displaying numbers or markers) and "hands" (typically two, indicating hours and minutes) but lacks any other human-like limbs such as arms or legs.


#### 4. Generate many outputs, then use the model to pick the best one

In [29]:
sytem_message = """
You are an AI assistant specialized in solving riddles.
Given a riddle, you have to generate three answers to the riddle.
For each answer, be specific about the reasoning you made.
Then, among the three answers, select the one that is most plausible given the riddle.

Riddle:

"""

riddle = """
What has a face and two hands, but no arms or legs?
"""

In [27]:
response = client.chat.completions.create(
  model="deepseek-v4-pro",
  messages=[
    {"role": "system", "content": system_message},
    {"role": "user", "content": riddle},
	],
  stream=False,
  reasoning_effort="high",
  extra_body={"thinking": {"type": "enabled"}}
)

In [28]:
print(
  response.choices[0].message.content
)

A clock or a watch.

Justification: The riddle describes something with a "face" (the clock face displaying numbers) and "two hands" (the hour and minute hands that point to the time), but specifically notes it has no arms or legs. This fits a timepiece perfectly, as its hands are attached directly to the face without any limbs.


#### 5. Repeat instructions at the end

In [34]:
system_message = """
You are a sentiment analyzer. You classify conversations into three categories: positive, negative or neutral.
Return only the sentiment, in lowercase and without punctuation.

Conversation:

"""

conversation = """
Customer: Hi, I need some help with my order.
AI agent: Hello, welcome to our online store. I'm an AI agent and I'm here to assist you.
Customer: I ordered a pair of shoes yesterday, but I haven't received a confirmation email yet. Can you check the status of my order?
AI agent: Sure, I can help you with that. Can you please provide me with your order number and email address?
Customer: Yes, my order number is 123456789 and my email is john.doe@example.com.
AI agent: Thank you. I have found your order in our system. It looks like your order is still being processed and it will be shipped soon. You should receive a confirmation email within 24 hours.
Customer: OK, thank you for the information. How long will it take for the shoes to arrive?
AI agent: You're welcome. According to our shipping policy, it will take about 3 to 5 business days for the shoes to arrive at your address. You can track your order online using the tracking number that will be sent to your email once your order is shipped.
Customer: Alright, sounds good. Thank you for your help.
AI agent: It's my pleasure. Is there anything else I can do for you today?
Customer: No, that's all. Have a nice day.
AI agent: Thank you for choosing our online store. Have a nice day too. Goodbye.
"""

In [35]:
response = client.chat.completions.create(
  model="deepseek-v4-pro",
  messages=[
    {"role": "system", "content": system_message},
    {"role": "user", "content": conversation},
	],
  stream=False,
  reasoning_effort="high",
  extra_body={"thinking": {"type": "enabled"}}
)

In [36]:
print(
  response.choices[0].message.content
)

neutral


---

In [37]:
sytem_message = f"""
You are a sentiment analyzer. You classify conversations into three categories: positive, negative or neutral.
Return only the sentiment, in lowercase and without punctuation.

Conversation:
{conversation}

Remember to return only the sentiment, in lowercase and without punctuation.
"""

In [38]:
response = client.chat.completions.create(
  model="deepseek-v4-pro",
  messages=[
    {"role": "user",
     "content": system_message},
	],
  stream=False,
  reasoning_effort="high",
  extra_body={"thinking": {"type": "enabled"}}
)

In [39]:
print(
  response.choices[0].message.content
)

neutral


---

#### 6. Use delimiters

In [40]:
system_message = """
You are a Python expert who produces Python code as per the user's request.

===>START EXAMPLE

---User Query---

Give me a function to print a string of text.

---User Output---
Below you can find the described function:
```
	def my_print(text):
		return print(text)
```

<===END EXAMPLE
"""

query = "generate a Python function to calculate the nth Fibonacci number"

In [41]:
response = client.chat.completions.create(
  model="deepseek-v4-pro",
  messages=[
    {"role": "system", "content": system_message},
    {"role": "user", "content": query},
	],
  stream=False,
  reasoning_effort="high",
  extra_body={"thinking": {"type": "enabled"}}
)

In [42]:
print(
  response.choices[0].message.content
)

Below you can find the described function:
```
def fibonacci(n):
    """
    Calculate the nth Fibonacci number (F(0) = 0, F(1) = 1).
    
    Args:
        n (int): Non-negative integer index.

    Returns:
        int: The nth Fibonacci number.
    
    Raises:
        ValueError: If n is negative.
    """
    if n < 0:
        raise ValueError("n must be a non-negative integer")
    if n == 0:
        return 0
    elif n == 1:
        return 1

    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b
```


### Advanced techniques

#### 1. Few-shot approach

In [49]:
system_message = """
You are an AI marketing assistant. You help users to create taglines for new product names.
Given a product name, produce a tagline similar to the following examples:

Peak Pursuit - Conquer Heights with Comfort
Summit Steps - Your Partner for Every Ascent
Crag Conquerors - Step Up, Stand Tall

Product name:

"""

product_name = "Elevation Embrace"

In [50]:
response = client.chat.completions.create(
  model="deepseek-v4-pro",
  messages=[
    {"role": "system", "content": system_message},
    {"role": "user", "content": product_name},
	],
  stream=False,
  reasoning_effort="high",
  extra_body={"thinking": {"type": "enabled"}}
)

In [51]:
print(
  response.choices[0].message.content
)

Elevation Embrace - Embrace Every Elevation


In [52]:
system_message = """
You are a binary classifier for sentiment analysis.
Given a text, based on its sentiment, you classify it into one of two categories: positive or negative.

You can use the following texts as examples:

Text: "I love this product! It's fantastic and works perfectly."
Positive

Text: "I'm really disappointed with the quality of the food."
Negative

Text: "This is the best day of my life!"
Positive

Text: "I can't stand the noise in this restaurant."
Negative

ONLY return the sentiment as output (without punctuation).

Text:

"""

In [53]:
import numpy as np
import pandas as pd

In [54]:
df = pd.read_csv('./data/movie.csv', encoding='utf-8')
df['label'] = df['label'].replace({0: 'Negative', 1: 'Positive'})
df.head()

,text,label
0,I grew up (b. 1965) watching and loving the Th...,Negative
1,"When I put this movie in my DVD player, and sa...",Negative
2,Why do people who do not know what a particula...,Negative
3,Even though I have great interest in Biblical ...,Negative
4,Im a die hard Dads Army fan and nothing will e...,Positive


In [58]:
# Test the performance of our model over a sample of 10 observations of this dataset
df = df.sample(n=10, random_state=42)

def process_text(text):
  response = client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=[
      {"role": "system", "content": system_message},
      {"role": "user", "content": text},
		],
    stream=False,
    reasoning_effort="high",
    extra_body={"thinking": {"type": "enabled"}}
	)
  return response.choices[0].message.content
  
df['predicted'] = df['text'].apply(process_text)

print(df)

                                                    text     label predicted
32823  The central theme in this movie seems to be co...  Negative  Negative
16298  An excellent example of "cowboy noir", as it's...  Positive  Positive
36572  Sure, we all like bad movies at one time or an...  Negative  Negative
6689   Only the chosen ones will appreciate the quali...  Positive  Positive
29591  This is one of those movies that has everythin...  Positive  Positive
28505  The ending made my heart jump up into my throa...  Negative  Positive
31067  This true story of Carlson's Raiders is more o...  Negative  Negative
26893  This is a really funny film, especially the se...  Positive  Positive
18948  i saw this film over 20 years ago and still re...  Positive  Positive
12335  Why?!! This was an insipid, uninspired and emb...  Negative  Negative


#### 2. Chain of thought

In [59]:
system_message = """
To solve a generic first-degree equation, follow these steps:

1. **Identify the Equation:** Start by identifying the equation you want to solve. It should be in the form of "ax + b = c", where 'a' is the coefficient of the variable, 'x' is the variable, 'b' is a constant and 'c' is another constant.

2. **Isolate the Variable:** Your goal is to isolate the variable 'x' on one side of the equation. To do this, perform the following steps:

	a. **Add or Subtract Constants:** Add or subtract 'b' from both sides of the equation to move constants to one side.
  
  b. **Divide by the Coefficient:** Divide both sides by 'a' to isolate 'x'. If 'a' is zero, the equation may not have a unique solution.
  
3. **Simplify:** Simplify both sides of the equation as much as possible.

4. **Solve for 'x':** Once 'x' is isolated on one side, you have the solution. It will be in the form of 'x = value'.

5. **Check Your Solution:** Plug the found value of 'x' back into the original equation to ensure it satisfies the equation. If it does, you've found the correct solution.

6. **Express the Solution:** Write down the solution in a clear and concise form.

7. **Consider Special Cases:** Be aware of special cases where there may be no solution or infinitely many solutions, especially if 'a' equals zero.

Equation:

"""

equation = "3x + 5 = 11"

In [154]:
response = client.chat.completions.create(
  model="deepseek-v4-pro",
  messages=[
    {"role": "system", "content": system_message},
    {"role": "user", "content": equation},
	],
  stream=False,
  reasoning_effort="high",
  extra_body={"thinking": {"type": "enabled"}}
)

In [155]:
print(
  response.choices[0].message.content
)

The solution to the equation \(3x + 5 = 11\) is:

1. Subtract 5 from both sides:  
   \(3x + 5 - 5 = 11 - 5\)  
   \(3x = 6\)

2. Divide both sides by 3:  
   \(\frac{3x}{3} = \frac{6}{3}\)  
   \(x = 2\)

**Check:** Substitute \(x = 2\) back into the original equation:  
\(3(2) + 5 = 6 + 5 = 11\), which is correct.

**Answer:** \(x = 2\)


#### 3. ReAct (Reason and Act)

In [124]:
from langchain.agents import create_agent
from langchain_deepseek import ChatDeepSeek
from langchain_tavily import TavilySearch

In [ ]:
# Initialize the llm
llm = ChatDeepSeek(
	model="deepseek-chat",
	temperature=0,
	max_tokens=None,
	max_retries=2,
	api_key=config.DEEPSEEK_API_KEY,
)

# Setup the web search tool
tavily_search = TavilySearch(
	max_results=5,
	topic="general",
	tavily_api_key=config.TAVILY_API_KEY,
)

# Build a smart LangChain agent & bind the web search tool
agent = create_agent(
	model=llm,
	tools=[tavily_search],
	system_prompt="""
	You are a helpful AI assistant.
	You conduct a websearch, a useful approach, when you need to answer questions about current events.
	"""
)

# Inspect the precompiled approach (LangSmith is used for this)
print(agent)

In [136]:
# Test the agent by asking something about the upcoming Olympic games
result = agent.invoke(
	{
		"messages": [
			{
				"role": "user",
				"content": "Who are going to be the Italian male athletes for climbing at the Los Angeles 2028 Olympics?"
			}
		]
	}
)

In [147]:
print(result["messages"][-1].pretty_print())

================================== Ai Message ==================================

Now I have a good picture. Let me compile the information.

## Italian Male Athletes for Climbing at the LA 2028 Olympics

**Important upfront note:** The qualification process for the Los Angeles 2028 Olympics hasn't happened yet — it will begin in **2027**. So no athletes are officially selected at this point. However, based on current top Italian male climbers, here are the strongest contenders:

### 🥇 Top Italian Male Contenders

**1. Stefano Ghisolfi** (Lead / Boulder)
- Italy's most accomplished male competition climber
- Multiple-time Italian Champion and World Cup medalist
- Competed at Paris 2024 in the Boulder & Lead combined event
- One of only a handful of climbers to have redpointed 9b+ (5.15c) outdoors
- Very strong candidate for the **Lead** discipline at LA 2028

**2. Filip Schenk** (Lead / Boulder)
- Rising Italian star who won his first Lead World Cup bronze in Chamonix 2025
- Also won t